# Tools and MCP

## Overview
Tools are how an agent touches the world. This notebook shows the two ways to give tools to a Strands agent, each with the smallest example that works:

1. **A custom tool**: a plain Python function with the `@tool` decorator.
2. **An MCP tool**: tools served by a [Model Context Protocol](https://modelcontextprotocol.io/introduction) server that the agent discovers at runtime.

The model provider and model come from `config.yaml`. For a fuller custom-tool agent (a SQLite appointment assistant) see [appointment-cli/appointment_agent.py](appointment-cli/appointment_agent.py), run from the command line.

## What it does
* Defines a custom tool and inspects the schema the model receives
* Shows how a tool error reaches the model
* Connects an agent to an MCP server over stdio and lets it use the server's tools

## Setup and prerequisites

* Python 3.10+, `pip install -r requirements.txt`
* The model from `config.yaml` available

In [1]:
#!pip install -r requirements.txt

In [2]:
import logging, warnings
import yaml
from strands import Agent, tool
from strands.models.ollama import OllamaModel

warnings.filterwarnings("ignore")
logging.getLogger("strands").setLevel(logging.WARNING)

cfg = yaml.safe_load(open("config.yaml"))

def make_model(**overrides):
    """One place to build the model from config.yaml."""
    params = dict(
        host=cfg["ollama"]["url"],
        model_id=cfg["ollama"]["models"][0],
        temperature=cfg["strands"]["temperature"],
        options={"num_ctx": cfg["strands"]["num_ctx"]},
    )
    params.update(overrides)
    return OllamaModel(**params)

model = make_model()
print("Model:", model.config["model_id"], "@", cfg["ollama"]["url"])

import json

Model: gemma4:e2b-mlx @ http://localhost:11434


## Part 1: A custom tool

Add `@tool` to a function. Strands reads the function name, type hints and docstring and builds the tool schema from them, so write the docstring for the model: what the tool does and what each argument means.

In [3]:
from typing import Literal

@tool
def convert_temperature(value: float, to: Literal["C", "F"]) -> str:
    """Convert a temperature between Celsius and Fahrenheit.

    Args:
        value: The temperature to convert.
        to: The target unit: "C" converts from Fahrenheit to Celsius, "F" converts from Celsius to Fahrenheit.
    """
    if to == "F":
        return f"{value} C = {value * 9 / 5 + 32:.1f} F"
    return f"{value} F = {(value - 32) * 5 / 9:.1f} C"

agent = Agent(
    model=model,
    tools=[convert_temperature],
    system_prompt="Use the convert_temperature tool for every conversion, then answer in one short sentence.",
)
print(json.dumps(agent.tool_registry.get_all_tools_config()["convert_temperature"], indent=2))

{
  "name": "convert_temperature",
  "description": "Convert a temperature between Celsius and Fahrenheit.",
  "inputSchema": {
    "json": {
      "properties": {
        "value": {
          "description": "The temperature to convert.",
          "type": "number"
        },
        "to": {
          "description": "The target unit: \"C\" converts from Fahrenheit to Celsius, \"F\" converts from Celsius to Fahrenheit.",
          "enum": [
            "C",
            "F"
          ],
          "type": "string"
        }
      },
      "required": [
        "value",
        "to"
      ],
      "type": "object"
    }
  }
}


That schema is what the model sees. Now ask a question that needs the tool:

In [4]:
response = agent("How many degrees Fahrenheit is 37 degrees Celsius?")


Tool #1: convert_temperature


37 degrees Celsius is 98.6 degrees Fahrenheit.

### When a tool fails

If a `@tool` raises, Strands catches it and returns a tool result with `status: "error"` and the message, instead of crashing the run. The model reads the error and can retry. Calling the tool directly shows the shape:

In [5]:
@tool
def get_item_price(item: str) -> float:
    """Look up the price of a menu item.

    Args:
        item: Name of the item: coffee, tea or cake.
    """
    prices = {"coffee": 3.5, "tea": 2.5, "cake": 4.0}
    if item not in prices:
        raise ValueError(f"Unknown item '{item}'. Choose from {list(prices)}.")
    return prices[item]

agent = Agent(model=model, tools=[get_item_price])
bad = agent.tool.get_item_price(item="pizza")
print("status :", bad["status"])
print("content:", bad["content"][0]["text"])

status : error
content: Error: Unknown item 'pizza'. Choose from ['coffee', 'tea', 'cake'].


## Part 2: An MCP tool

Instead of writing the tool in your agent's process, you connect to an **MCP server** that exposes tools. The agent discovers them at runtime. With the **stdio** transport the client starts the server as a child process and talks to it over stdin/stdout.

<div style="text-align:center">
    <img src="images/mcp_stdio.svg" width="75%" />
</div>

The course math server in [mcp/servers/math_server.py](mcp/servers/math_server.py) exposes `add`, `subtract`, `multiply`, `divide` and `power`. `MCPClient` takes a function that builds the transport. We launch the server with the same Python interpreter as this notebook.

In [6]:
import sys
from pathlib import Path
from mcp import StdioServerParameters, stdio_client
from strands.tools.mcp import MCPClient

SERVERS = Path("mcp/servers").resolve()

stdio_math = MCPClient(lambda: stdio_client(
    StdioServerParameters(command=sys.executable, args=[str(SERVERS / "math_server.py")])
))

Keep the client open (`with`) while the agent runs, because the tools call back into the server. `list_tools_sync()` discovers the tools:

In [7]:
with stdio_math:
    tools = stdio_math.list_tools_sync()
    print("Discovered:", [t.tool_name for t in tools])

    agent = Agent(model=model, tools=tools,
                  system_prompt="Use the math tools for every calculation, then state the result briefly.")
    response = agent("What is 125 plus 375?")

Discovered: ['add', 'subtract', 'multiply', 'divide', 'power']



Tool #1: add


500

## Part 3: A remote MCP server

Stdio starts the server on your machine. A **remote** server is already running somewhere else and you connect over **Streamable HTTP** with just a URL. [DeepWiki](https://mcp.deepwiki.com) is a free public server (no API key) that answers questions about any public GitHub repository. Only the transport changes: the `MCPClient` and the agent code stay the same.

In [8]:
from mcp.client.streamable_http import streamablehttp_client

deepwiki = MCPClient(lambda: streamablehttp_client("https://mcp.deepwiki.com/mcp"))

with deepwiki:
    all_tools = deepwiki.list_tools_sync()
    print("Discovered:", [t.tool_name for t in all_tools])

    # skip read_wiki_contents: it returns a whole wiki, too large for a small model's context
    tools = [t for t in all_tools if t.tool_name != "read_wiki_contents"]
    agent = Agent(model=model, tools=tools,
                  system_prompt="Use the DeepWiki tools to answer questions about GitHub repositories. Answer in two sentences.")
    response = agent("In the strands-agents/sdk-python repository, what does the Agent class do?")

Discovered: ['ask_wiki_question', 'read_wiki_contents', 'read_wiki_structure']



Tool #1: ask_wiki_question


The `Agent` class is the core orchestrator in the

 Strands Agents SDK, responsible for managing interactions between language models, tools, and conversation history to process user

 input and generate responses. It coordinates the workflow by handling model interaction,

 tool execution, and maintaining conversation context to produce final results.

Several MCP servers can feed one agent, and the servers in [mcp/servers/](mcp/servers/) also support Streamable HTTP (see [mcp/README.md](mcp/README.md)). Exercise 7.1 builds a server of your own.

## Summary

1. A custom tool is a function with `@tool`. Its **type hints and docstring are its schema**.
2. A tool that raises is reported to the model as an error result, not a crash.
3. An MCP server provides tools outside your process. `MCPClient` connects and `list_tools_sync()` discovers them. It can be local (stdio) or remote (Streamable HTTP, Part 3).
4. Both run on the model chosen in `config.yaml`. Next: memory (7.3).